In [3]:
import json
import pandas as pd
from pprint import pprint
import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns

Load the JSON and create a master DataFrame of STIX objects

In [4]:
enterprise_file = "../enterprise-attack/enterprise-attack.json"
mobile_file = "../mobile-attack/mobile-attack.json"
ics_file = "../ics-attack/ics-attack.json"

with open(enterprise_file, "r", encoding="utf-8") as f:
    stix = json.load(f)

objects = stix.get("objects", [])
# Master dataframe (keeps the raw dict for reference)
master_df = pd.DataFrame([{
    "id": o.get("id"),
    "type": o.get("type"),
    "name": o.get("name"),
    "description": o.get("description"),
    "created": o.get("created"),
    "modified": o.get("modified"),
    "raw": o
} for o in objects])

master_df.head()


,id,type,name,description,created,modified,raw
0,x-mitre-collection--1f5f1533-f617-4ca8-9ab4-6a...,x-mitre-collection,Enterprise ATT&CK,ATT&CK for Enterprise provides a knowledge bas...,2018-01-17T12:56:55.080Z,2025-10-28T14:00:00.188Z,"{'type': 'x-mitre-collection', 'id': 'x-mitre-..."
1,x-mitre-matrix--eafc1b4c-5e56-4965-bd4e-66a6a8...,x-mitre-matrix,Enterprise ATT&CK,Below are the tactics and technique representi...,2018-10-17T00:14:20.652Z,2025-04-25T14:41:40.982Z,"{'type': 'x-mitre-matrix', 'spec_version': '2...."
2,course-of-action--00d7d21b-69d6-4797-88a2-c86f...,course-of-action,Password Filter DLL Mitigation,Ensure only valid password filters are registe...,2018-10-17T00:14:20.652Z,2025-04-18T17:59:39.912Z,"{'type': 'course-of-action', 'spec_version': '..."
3,course-of-action--02f0f92a-0a51-4c94-9bda-6437...,course-of-action,Space after Filename Mitigation,Prevent files from having a trailing space aft...,2018-10-17T00:14:20.652Z,2025-04-18T17:59:40.127Z,"{'type': 'course-of-action', 'spec_version': '..."
4,course-of-action--03c0c586-50ed-45a7-95f4-f496...,course-of-action,HISTCONTROL Mitigation,Prevent users from changing the <code>HISTCONT...,2018-10-17T00:14:20.652Z,2025-04-18T17:59:40.291Z,"{'type': 'course-of-action', 'spec_version': '..."


Filter for Intrusion Sets


In [12]:
# Filter only intrusion-set objects
apts_df = master_df[master_df["type"] == "intrusion-set"].copy()

print(f"Found {len(apts_df)} intrusion sets (APT groups).")
apts_df.head()

# Extract readable columns for APT analysis
apts_df["aliases"] = apts_df["raw"].apply(
    lambda o: ", ".join(o.get("aliases", [])) if isinstance(o, dict) else ""
)
apts_df["url"] = apts_df["raw"].apply(
    lambda o: next(
        (ref.get("url") for ref in o.get("external_references", []) if "url" in ref),
        None
    )
)

apt_text_df = apts_df[["id", "name", "aliases", "description", "url"]].reset_index(drop=True)
apt_text_df.head(10)

Found 187 intrusion sets (APT groups).


,id,name,aliases,description,url
0,intrusion-set--01e28736-2ffc-455b-9880-ed4d1407ae07,Indrik Spider,"Indrik Spider, Evil Corp, Manatee Tempest, DEV-0243, UNC2165","[Indrik Spider](https://attack.mitre.org/groups/G0119) is a Russia-based cybercriminal group that has been active since at least 2014. [Indrik Spider](https://attack.mitre.org/groups/G0119) initially started with the [Dridex](https://attack.mitre.org/software/S0384) banking Trojan, and then by 2017 they began running ransomware operations using [BitPaymer](https://attack.mitre.org/software/S0570), [WastedLocker](https://attack.mitre.org/software/S0612), and Hades ransomware. Following U.S. sanctions and an indictment in 2019, [Indrik Spider](https://attack.mitre.org/groups/G0119) changed their tactics and diversified their toolset.(Citation: Crowdstrike Indrik November 2018)(Citation: Crowdstrike EvilCorp March 2021)(Citation: Treasury EvilCorp Dec 2019)",https://attack.mitre.org/groups/G0119
1,intrusion-set--b7f627e2-0817-4cd5-8d50-e75f8aa85cc6,LuminousMoth,LuminousMoth,"[LuminousMoth](https://attack.mitre.org/groups/G1014) is a Chinese-speaking cyber espionage group that has been active since at least October 2020. [LuminousMoth](https://attack.mitre.org/groups/G1014) has targeted high-profile organizations, including government entities, in Myanmar, the Philippines, Thailand, and other parts of Southeast Asia. Some security researchers have concluded there is a connection between [LuminousMoth](https://attack.mitre.org/groups/G1014) and [Mustang Panda](https://attack.mitre.org/groups/G0129) based on similar targeting and TTPs, as well as network infrastructure overlaps.(Citation: Kaspersky LuminousMoth July 2021)(Citation: Bitdefender LuminousMoth July 2021)",https://attack.mitre.org/groups/G1014
2,intrusion-set--918da025-04bd-48af-b6c4-f3e4d1b915eb,Medusa Group,Medusa Group,"[Medusa Group](https://attack.mitre.org/groups/G1051) has been active since at least 2021 and was initially operated as a closed ransomware group before evolving into a Ransomware-as-a-Service (RaaS) operation. Some reporting indicates that certain attacks may still be conducted directly by the ransomware’s core developers. Public sources have also referred to the group as “Spearwing” or “Medusa Actors.” (Citation: CISA Medusa Group Medusa Ransomware March 2025) (Citation: Broadcom Medusa Ransomware Medusa Group March 2025) [Medusa Group](https://attack.mitre.org/groups/G1051) employs living-off-the-land techniques, frequently leveraging publicly available tools and common remote management software to conduct operations. The group engages in double extortion tactics, exfiltrating data prior to encryption and threatening to publish stolen information if ransom demands are not met. (Citation: Security Scorecard Medusa Ransomware January 2024) For initial access, [Medusa Group](https://attack.mitre.org/groups/G1051) has exploited publicly known vulnerabilities, conducted phishing campaigns, and used credentials or access purchased from Initial Access Brokers (IABs). The group is opportunistic and has targeted a wide range of sectors globally. (Citation: Intel471 Medusa Ransomware May 2025)",https://attack.mitre.org/groups/G1051
3,intrusion-set--dd2d9ca6-505b-4860-a604-233685b802c7,Wizard Spider,"Wizard Spider, UNC1878, TEMP.MixMaster, Grim Spider, FIN12, GOLD BLACKBURN, ITG23, Periwinkle Tempest, DEV-0193","[Wizard Spider](https://attack.mitre.org/groups/G0102) is a Russia-based financially motivated threat group originally known for the creation and deployment of [TrickBot](https://attack.mitre.org/software/S0266) since at least 2016. [Wizard Spider](https://attack.mitre.org/groups/G0102) possesses a diverse arsenal of tools and has conducted ransomware campaigns against a variety of organizations, ranging from major corporations to hospitals.(Citation: CrowdStrike Ryuk January 2019)(Citation: DHS/CISA Ransomware Targeting Healthcare October 2020)(Citation: CrowdStrike Wizard Spider October 

Export the data

In [7]:
apt_text_df.to_csv("apt_descriptions.csv", index=False)

Load in new data

## Analyses and Plots